# Baby Food Science — Data Pipeline

Run each section in order:
1. **Import** papers from OpenAlex into SQLite databases
2. **Enrich** with AI metadata (requires local Ollama)
3. **Build** universe.json and topic JSON for the frontend


In [ ]:
import subprocess, sys, os

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
print('Backend dir:', BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)

## Step 1 — Import papers from OpenAlex

Fetches papers for each topic and stores them in `../data/papers_<topic>.db`.

Adjust `TOPICS` and `MAX_PAPERS` as needed. First run with 50–100 to test.

In [ ]:
from import_openalex import import_topic, PREDEFINED_TOPICS

# Topics to import — start small
TOPICS = [
    'peanut_allergy',
    'complementary_feeding',
    'breastfeeding',
]
MAX_PAPERS = 100

for topic in TOPICS:
    print(f'\n=== Importing: {topic} ===')
    import_topic(topic, max_results=MAX_PAPERS)

In [ ]:
# Or import ALL predefined topics at once (takes longer)
# for topic in PREDEFINED_TOPICS:
#     import_topic(topic, max_results=200)

## Step 2 — Enrich with AI metadata

Requires Ollama running locally:
```
ollama serve
ollama pull mistral
```

This fills in `AI_primary_field`, `recommendation_summary`, `evidence_strength`, `likelihood_score`, etc.

In [ ]:
import glob
from process_ai import process_db, get_client

MODEL = 'mistral'  # or 'llama3', 'phi3', etc.
client, model = get_client(model=MODEL)

data_dir = os.path.join(BACKEND_DIR, '..', 'data')
dbs = sorted(glob.glob(os.path.join(data_dir, 'papers_*.db')))
print(f'Found {len(dbs)} databases')

for db_path in dbs:
    process_db(db_path, client, model, batch_size=10)

## Step 3 — Build universe.json and topic JSON

Generates the JSON files the frontend reads.

In [ ]:
from build_data import build_universe

data_dir = os.path.abspath(os.path.join(BACKEND_DIR, '..', 'data'))
out_dir  = os.path.abspath(os.path.join(BACKEND_DIR, '..', 'frontend', 'public'))

print('Data dir:', data_dir)
print('Out dir: ', out_dir)

build_universe(data_dir, out_dir)
print('\nDone! Start the frontend with: cd frontend && npm start')

## Inspect databases

In [ ]:
import sqlite3, glob, os

data_dir = os.path.abspath(os.path.join(BACKEND_DIR, '..', 'data'))
for db_path in sorted(glob.glob(os.path.join(data_dir, 'papers_*.db'))):
    conn = sqlite3.connect(db_path)
    total = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]
    classified = conn.execute("SELECT COUNT(*) FROM papers WHERE AI_primary_field IS NOT NULL").fetchone()[0]
    conn.close()
    name = os.path.basename(db_path)
    print(f'{name:45s}  {total:4d} papers  {classified:4d} classified')